## Laboratorio 6

- Diego Valenzuela 22309 
- Daniel Dubon 22233
- Nelson García Bravatti 22434
- Joaquin Puente 22296

# Task 1

1. El entorno de simulación que usarán es LunarLanderContinuous-v2 de Gymnasium, que tiene un
espacio de acción continuo de dos dimensiones representando la fuerza de dos propulsores.
Argumenten formalmente por qué Q-Learning tabular y DQN son inapropiados para este entorno. Su
argumento debe mencionar explícitamente el espacio de acción, el operador argmax, y la
representación de la política

Tanto Q-Learning tabular como DQN son algoritmos diseñados para entornos discretos por lo que su aplicación en LunarLanderContinuous-v2 es inapropiada ya que el entorno posee un espacio de acción continuo bidimensional para controlar las fuerzas de los dos propulsores. Q-Learning tabular requiere construir una tabla o matriz donde cada par de estado y acción tiene una entrada unica, como el espacio de acción es continuo existen infinitas acciones posibles y crear esta tabla es imposible sin aplicar una discretización extrema que causaria la maldición de la dimensionalidad y perdida de información. Para el caso de DQN el obstaculo principal es el operador argmax donde DQN usa redes neuronales y puede manejar estados continuos pero su ecuación de actualización de Bellman requiere aplicar el argmax sobre todas las acciones posibles para encontrar el mayor valor esperado. Evaluar el $\arg\max_a Q(s, a)$ en un espacio de acción continuo requiere resolver un problema de optimización complejo en cada iteración del agente lo cual hace que el algoritmo sea computacionalmente inviable.Finalmente esto se reduce a la representación de la politica. Los algoritmos basados en valor como Q-Learning y DQN tienen una representación de politica implicita que depende totalmente de evaluar acciones discretas individuales para elegir la mejor. En dominios de control continuo se necesitan algoritmos como Actor-Critic o REINFORCE que mantienen una representación de politica parametrizada y explicita.


---

2. Para LunarLanderContinuous-v2, la política se parametrizará como una distribución Gaussiana
𝜋𝜃(𝑎 ∣ 𝑠) = 𝒩(𝜇𝜃, (𝑠)𝜎2𝐼) donde 𝜇𝜃(𝑠) es la salida de una red neuronal. Expliquen cómo se calcula
∇𝜃 ln 𝜋𝜃 (𝐴𝑡 ∣ 𝑆𝑡) para esta parametrización específica. Desarrollen la expresión analítica del
gradiente del logaritmo de la densidad Gaussiana respecto a 𝜃, identificando qué parte depende de
𝜃 y qué parte no



Para calcular el gradiente del logaritmo de la politica primero debemos partir de la función de densidad de probabilidad de una distribución Gaussiana como la politica es $\pi_\theta(A_t \mid S_t) = \mathcal{N}(\mu_\theta(S_t), \sigma^2 I)$ su función de densidad se escribe de la siguiente manera:

$$\pi_\theta(A_t \mid S_t) = \frac{1}{\sqrt{(2\pi)^k \vert{}\sigma^2 I\vert{}}} \exp\left(-\frac{1}{2\sigma^2} \Vert{}A_t - \mu_\theta(S_t)\Vert{}^2\right)$$

Ahora aplicamos el logaritmo natural a esta formula para separar todo y el logaritmo convierte la multiplicación en suma y elimina el exponencial de la ecuación:

$$\ln \pi_\theta(A_t \mid S_t) = -\frac{k}{2} \ln(2\pi \sigma^2) - \frac{1}{2\sigma^2} \Vert{}A_t - \mu_\theta(S_t)\Vert{}^2$$

El siguiente paso es derivar esta formula con respecto a los parametros $\theta$. Al aplicar el gradiente $\nabla_\theta$ vemos que el primer termino de la normalización es una constante que no depende de $\theta$ por lo que su derivada es cero. Solo derivamos el termino de la derecha aplicando la regla de la cadena para obtener la expresión analitica final:

$$\nabla_\theta \ln \pi_\theta(A_t \mid S_t) = \frac{(A_t - \mu_\theta(S_t))}{\sigma^2} \nabla_\theta \mu_\theta(S_t)$$

Al observar este resultado podemos identificar claramente las dependencias:

Las partes que dependen de $\theta$ son la media $\mu_\theta(S_t)$ ya que es la salida directa de la red neuronal y el jacobiano $\nabla_\theta \mu_\theta(S_t)$ que representa los gradientes de la red neuronal respecto a sus propios pesos.

Las partes que no dependen de $\theta$ son la varianza $\sigma^2$ (que en esta parametrización especifica es un escalar constante que no se aprende) la acción muestreada $A_t$ y el estado del entorno $S_t$. La constante inicial de la distribución tampoco depende de $\theta$ y por eso desaparece durante la derivación.

---

3. Comparen formalmente REINFORCE con línea base y Actor-Critic en términos de sesgo y varianza del
estimador del gradiente. Para cada algoritmo identifiquen: qué usa como estimador de la ventaja 𝐴̂ 𝑡
, qué componente introduce sesgo, y qué componente introduce varianza. Predigan cuál algoritmo
esperan que converja más rápido en LunarLanderContinuous-v2 y justifiquen esa predicción.

REINFORCE con linea base utiliza el retorno real del episodio menos la predicción de la linea base como estimador de la ventaja $\hat{A}_t = G_t - V(S_t)$. En este algoritmo el estimador no tiene sesgo porque el retorno $G_t$ es una muestra real y exacta obtenida del entorno, sin embargo el componente que introduce alta varianza es justamente el uso de $G_t$ ya que calcular el retorno requiere sumar las recompensas de toda una trayectoria completa acumulando la aleatoriedad e incertidumbre de cada paso hasta el final pero por otro lado Actor-Critic utiliza el error de diferencia temporal como estimador de la ventaja $\hat{A}_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$. El componente que introduce sesgo es el uso de $V(S_{t+1})$ lo cual se conoce como bootstrapping. Al usar la predicción de la propia red neuronal del critico para estimar el futuro en lugar de usar datos reales se introduce un sesgo fuerte especialmente al inicio de la simulación cuando la red aun no sabe nada y a cambio el componente que elimina la varianza es el hecho de usar un solo paso real $R_{t+1}$ evitando toda la incertidumbre de una trayectoria completa.Para el entorno LunarLanderContinuous-v2 se prevee que Actor-Critic convergera mas rapido ya que en un espacio de acción continuo las trayectorias tienen una varianza inmensa porque existen infinitas formas de activar los propulsores y REINFORCE sufre demasiado con esta varianza y sus actualizaciones de gradiente serian muy caoticas e inestables. Actor-Critic sacrifica precisión inicial aceptando el sesgo pero al actualizar la politica en cada paso con varianza baja logra estabilizar el entrenamiento mucho mas pronto y aprender a aterrizar en una fracción del tiempo.

---

4. El entorno de exoesqueleto real tiene una restricción que LunarLanderContinuous-v2 no tiene:
las acciones deben ser suaves en el tiempo para no causar movimientos bruscos que dañen al
paciente. Argumenten cómo modificarían la función de recompensa y la parametrización de la
política para incorporar esa restricción. ¿Cambiaría eso la elección entre REINFORCE y Actor-Critic?

Para modificar la función de recompensa se agregaria un termino de penalización que castigue los cambios bruscos en las fuerzas del motor restando a la recompensa original un valor proporcional a la diferencia matematica entre la acción actual y la acción anterior. De esta forma el agente recibe menos puntos si el exoesqueleto da tirones o cambia de fuerza de un instante a otro por lo que el algoritmo se ve forzado a priorizar un movimiento fluido.

En cuanto a la parametrización de la politica la red neuronal ya no deberia calcular la fuerza absoluta del motor de forma aislada. La forma mas inteligente de incorporar esta restricción es hacer que la red neuronal calcule solamente el cambio de fuerza o delta y que ese pequeño ajuste se sume a la acción anterior u otra opcion es incluir la acción pasada directamente como parte del estado de entrada para que el modelo tome la siguiente decisión teniendo ese contexto en cuenta.

Esta modificación no cambiaria la elección entre REINFORCE y Actor-Critic sino que haria a Actor-Critic mucho mas importante porque al agregar la acción anterior al sistema y requerir suavidad en el tiempo el problema de control se vuelve mas estricto y evaluar episodios completos generaria una varianza todavia mas alta lo que destruiria el gradiente en REINFORCE. Actor-Critic sigue siendo la mejor opción porque su capacidad de estimar la ventaja paso a paso mantiene el aprendizaje estable y permite que la red se adapte a estas restricciones sin sufrir por el ruido de toda la trayectoria.

# Task 2

Implementación desde cero de REINFORCE con línea base y Actor-Critic para `LunarLanderContinuous-v2`. Ambos métodos usan exactamente la arquitectura y las métricas solicitadas.

## 1. Dependencias y configuración

La siguiente celda instala en Google Colab una versión de Gymnasium que incluye `LunarLanderContinuous-v2`. Después de instalar, si Colab lo solicita, se debe reiniciar la sesión y continuar desde la celda de importaciones.

In [ ]:
%pip install -q swig
%pip install -q "gymnasium[box2d]==0.29.1"

In [ ]:
import random
from collections import defaultdict

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import HTML
from matplotlib import animation
from torch.distributions import Normal

ENV_ID = "LunarLanderContinuous-v2"
SEED = 42
NUM_EPISODES = 1000
GAMMA = 0.99
ACTOR_LR = 3e-4
CRITIC_LR = 1e-3
ENTROPY_COEF = 1e-3
VALUE_COEF = 0.5
MAX_GRAD_NORM = 1.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Dispositivo: {DEVICE}")

## 2. Redes y funciones auxiliares

La política produce $\mu_\theta(s)$ mediante dos capas ocultas de 64 neuronas con ReLU y una salida `tanh`. El vector `log_std` es un parámetro aprendible independiente del estado e inicia en $\log(0.5)$. La acción muestreada se recorta únicamente al enviarla al entorno; el `log_prob` se calcula sobre la muestra gaussiana original. En REINFORCE, la línea base es una red independiente. En Actor-Critic, el torso de dos capas es compartido y se divide en una cabeza para la media y otra cabeza escalar para $\hat V_w(s)$.

In [ ]:
def init_layer(layer, gain=np.sqrt(2)):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.zeros_(layer.bias)
    return layer

class GaussianPolicy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            init_layer(nn.Linear(state_dim, 64)), nn.ReLU(),
            init_layer(nn.Linear(64, 64)), nn.ReLU(),
            init_layer(nn.Linear(64, action_dim), gain=0.01), nn.Tanh(),
        )
        self.log_std = nn.Parameter(torch.full((action_dim,), np.log(0.5)))

    def distribution(self, state):
        mean = self.net(state)
        std = self.log_std.clamp(-5.0, 2.0).exp().expand_as(mean)
        return Normal(mean, std)

class ValueNetwork(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.net = nn.Sequential(
            init_layer(nn.Linear(state_dim, 64)), nn.ReLU(),
            init_layer(nn.Linear(64, 64)), nn.ReLU(),
            init_layer(nn.Linear(64, 1), gain=1.0),
        )

    def forward(self, state):
        return self.net(state).squeeze(-1)

class SharedActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.body = nn.Sequential(
            init_layer(nn.Linear(state_dim, 64)), nn.ReLU(),
            init_layer(nn.Linear(64, 64)), nn.ReLU(),
        )
        self.mean_head = init_layer(nn.Linear(64, action_dim), gain=0.01)
        self.value_head = init_layer(nn.Linear(64, 1), gain=1.0)
        self.log_std = nn.Parameter(torch.full((action_dim,), np.log(0.5)))

    def forward(self, state):
        features = self.body(state)
        mean = torch.tanh(self.mean_head(features))
        value = self.value_head(features).squeeze(-1)
        std = self.log_std.clamp(-5.0, 2.0).exp().expand_as(mean)
        return Normal(mean, std), value

    def actor_parameters(self):
        return list(self.body.parameters()) + list(self.mean_head.parameters()) + [self.log_std]

def discounted_returns(rewards, gamma):
    returns, running_return = [], 0.0
    for reward in reversed(rewards):
        running_return = reward + gamma * running_return
        returns.append(running_return)
    return torch.tensor(returns[::-1], dtype=torch.float32, device=DEVICE)

def gradient_norm(parameters):
    squared_norm = sum(
        parameter.grad.detach().pow(2).sum()
        for parameter in parameters if parameter.grad is not None
    )
    return float(torch.sqrt(squared_norm).cpu()) if not isinstance(squared_norm, int) else 0.0

def norm_from_gradients(gradients):
    valid = [gradient.detach().pow(2).sum() for gradient in gradients if gradient is not None]
    return float(torch.sqrt(torch.stack(valid).sum()).cpu()) if valid else 0.0

def empty_history():
    return defaultdict(list)

## 3. REINFORCE con línea base

Al terminar cada episodio se calculan los retornos Monte Carlo $G_t$. La ventaja es $\hat A_t=G_t-V_\phi(S_t)$; el retorno se separa del grafo al entrenar la línea base y la ventaja se separa al actualizar el Actor.

In [ ]:
def train_reinforce(num_episodes=NUM_EPISODES, seed=SEED):
    set_seed(seed)
    env = gym.make(ENV_ID)
    env.action_space.seed(seed)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    policy = GaussianPolicy(state_dim, action_dim).to(DEVICE)
    baseline = ValueNetwork(state_dim).to(DEVICE)
    actor_optimizer = torch.optim.Adam(policy.parameters(), lr=ACTOR_LR)
    baseline_optimizer = torch.optim.Adam(baseline.parameters(), lr=CRITIC_LR)
    history = empty_history()

    for episode in range(num_episodes):
        state, _ = env.reset(seed=seed + episode)
        states, rewards, log_probs, entropies = [], [], [], []
        terminated = truncated = False

        while not (terminated or truncated):
            state_tensor = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
            distribution = policy.distribution(state_tensor)
            sampled_action = distribution.sample()
            env_action = sampled_action.clamp(-1.0, 1.0).cpu().numpy()
            next_state, reward, terminated, truncated, _ = env.step(env_action)
            states.append(state_tensor)
            rewards.append(float(reward))
            log_probs.append(distribution.log_prob(sampled_action).sum())
            entropies.append(distribution.entropy().sum())
            state = next_state

        states_tensor = torch.stack(states)
        returns = discounted_returns(rewards, GAMMA)
        values = baseline(states_tensor)
        advantages = returns - values

        actor_loss = -(torch.stack(log_probs) * advantages.detach()).mean()
        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_grad_norm = gradient_norm(policy.parameters())
        nn.utils.clip_grad_norm_(policy.parameters(), MAX_GRAD_NORM)
        actor_optimizer.step()

        baseline_loss = nn.functional.mse_loss(values, returns)
        baseline_optimizer.zero_grad()
        baseline_loss.backward()
        nn.utils.clip_grad_norm_(baseline.parameters(), MAX_GRAD_NORM)
        baseline_optimizer.step()

        history["reward"].append(sum(rewards))
        history["actor_grad_norm"].append(actor_grad_norm)
        history["entropy"].append(float(torch.stack(entropies).mean().detach().cpu()))
        if (episode + 1) % 50 == 0:
            mean_reward = np.mean(history["reward"][-20:])
            print(f"REINFORCE | episodio {episode + 1:4d} | recompensa media(20): {mean_reward:8.2f}")

    env.close()
    return policy, baseline, dict(history)

reinforce_policy, reinforce_baseline, reinforce_history = train_reinforce()

## 4. Actor-Critic de un paso

El estimador de ventaja es el error TD $\delta_t=r_t+\gamma(1-d_t)V_w(s_{t+1})-V_w(s_t)$. Se actualiza en cada transición. Para reportar $\lVert\nabla_\theta L\rVert$, se mide por separado el gradiente de la pérdida del Actor sobre el torso compartido, la cabeza de la media y `log_std`, antes de sumar la pérdida del Critic.

In [ ]:
def train_actor_critic(num_episodes=NUM_EPISODES, seed=SEED):
    set_seed(seed)
    env = gym.make(ENV_ID)
    env.action_space.seed(seed)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    model = SharedActorCritic(state_dim, action_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=ACTOR_LR)
    history = empty_history()

    for episode in range(num_episodes):
        state, _ = env.reset(seed=seed + episode)
        episode_reward, step_grad_norms, step_entropies = 0.0, [], []
        terminated = truncated = False

        while not (terminated or truncated):
            state_tensor = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
            distribution, value = model(state_tensor)
            sampled_action = distribution.sample()
            log_prob = distribution.log_prob(sampled_action).sum()
            entropy = distribution.entropy().sum()
            env_action = sampled_action.clamp(-1.0, 1.0).cpu().numpy()
            next_state, reward, terminated, truncated, _ = env.step(env_action)
            done = terminated or truncated

            with torch.no_grad():
                if done:
                    next_value = torch.zeros((), device=DEVICE)
                else:
                    next_state_tensor = torch.as_tensor(next_state, dtype=torch.float32, device=DEVICE)
                    _, next_value = model(next_state_tensor)
                td_target = torch.as_tensor(reward, dtype=torch.float32, device=DEVICE) + GAMMA * next_value

            advantage = td_target - value
            actor_loss = -log_prob * advantage.detach() - ENTROPY_COEF * entropy
            critic_loss = 0.5 * advantage.pow(2)
            actor_parameters = model.actor_parameters()
            actor_gradients = torch.autograd.grad(
                actor_loss, actor_parameters, retain_graph=True, allow_unused=True
            )
            step_grad_norms.append(norm_from_gradients(actor_gradients))

            optimizer.zero_grad()
            (actor_loss + VALUE_COEF * critic_loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

            episode_reward += float(reward)
            step_entropies.append(float(entropy.detach().cpu()))
            state = next_state

        history["reward"].append(episode_reward)
        history["actor_grad_norm"].append(float(np.mean(step_grad_norms)))
        history["entropy"].append(float(np.mean(step_entropies)))
        if (episode + 1) % 50 == 0:
            mean_reward = np.mean(history["reward"][-20:])
            print(f"Actor-Critic | episodio {episode + 1:4d} | recompensa media(20): {mean_reward:8.2f}")

    env.close()
    return model, dict(history)

actor_critic_model, actor_critic_history = train_actor_critic()

## 5. Resultados y cuatro gráficas solicitadas

La primera gráfica compara las curvas de aprendizaje con media móvil de 20 episodios. Las siguientes muestran recompensa sin suavizar, norma del gradiente del Actor y entropía de la política.

In [ ]:
def moving_average(values, window=20):
    values = np.asarray(values, dtype=np.float64)
    if len(values) < window:
        return np.arange(1, len(values) + 1), values
    averaged = np.convolve(values, np.ones(window) / window, mode="valid")
    return np.arange(window, len(values) + 1), averaged

histories = {
    "REINFORCE + línea base": reinforce_history,
    "Actor-Critic": actor_critic_history,
}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for label, history in histories.items():
    episodes_ma, rewards_ma = moving_average(history["reward"], 20)
    axes[0, 0].plot(episodes_ma, rewards_ma, label=label)
    axes[0, 1].plot(history["reward"], alpha=0.65, label=label)
    axes[1, 0].plot(history["actor_grad_norm"], alpha=0.8, label=label)
    axes[1, 1].plot(history["entropy"], alpha=0.8, label=label)

titles = [
    "Curvas de aprendizaje (media móvil de 20)",
    "Recompensa total por episodio",
    r"Norma del gradiente del Actor $\|\nabla_\theta L\|$",
    r"Entropía media de la política $\mathbb{E}[\mathcal{H}(\pi_\theta)]$",
]
for axis, title in zip(axes.flat, titles):
    axis.set_title(title)
    axis.set_xlabel("Episodio")
    axis.grid(alpha=0.25)
    axis.legend()
axes[0, 0].set_ylabel("Recompensa media")
axes[0, 1].set_ylabel("Recompensa")
axes[1, 0].set_ylabel("Norma L2")
axes[1, 1].set_ylabel("Entropía")
plt.tight_layout()
plt.show()

## 6. Visualización de un episodio aprendido

Se usa la media de la política Actor-Critic para una evaluación determinista. El entorno se crea con `render_mode="rgb_array"` y cada cuadro se obtiene mediante `env.render()`.

In [ ]:
def render_learned_episode(model, seed=SEED + 10_000):
    env = gym.make(ENV_ID, render_mode="rgb_array")
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0
    terminated = truncated = False
    model.eval()

    while not (terminated or truncated):
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)
            distribution, _ = model(state_tensor)
            action = distribution.mean.clamp(-1.0, 1.0).cpu().numpy()
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += float(reward)
        frames.append(env.render())

    env.close()
    model.train()
    print(f"Recompensa del episodio visualizado: {total_reward:.2f}")

    figure = plt.figure(figsize=(8, 6))
    plt.axis("off")
    artists = [[plt.imshow(frame, animated=True)] for frame in frames]
    video = animation.ArtistAnimation(figure, artists, interval=30, blit=True)
    plt.close(figure)
    return HTML(video.to_jshtml())

render_learned_episode(actor_critic_model)